# From Reviews to Insights: Google Play Review Pipeline for Indonesian Mobile Banking Apps

**Apps:** BRImo, myBCA, Livin' by Mandiri, wondr by BNI  
**Goal:** a clean, reproducible dataset for fine-tuning IndoBERT on three-class sentiment (negative / neutral / positive).

| Stage | Purpose | Main output |
|---|---|---|
| 1. Scraping | Collect every review in the study period | `data/raw/<app>_raw.csv`, scrape logs |
| 2. Audit and population | Check coverage and quality, build the study population | `data/interim/population_period.csv` |
| 3. Text normalization | Add normalized text columns and quality flags | `data/processed/population_normalized.csv` |
| 4. Sampling for labeling | Draw test and training-pool samples, build a blind labeling sheet | `data/labeling/labeling_sheet_<batch>.csv`, `sample_key_<batch>.csv` |
| 5. Labeling guidelines | Rules for manual labeling | reference only |
| 6. Label validation | Consistency checks on the labeled sheet, agreement statistics | validated labeled data |
| 7. Training data assembly | Fold assignment, final training and test files | `data/processed/train_pool.csv`, `test.csv` |

**Design principles**

- Raw data is stored untouched. Every later step writes new columns or new files.
- The labeled sample exists only to train and evaluate the model. The trained model later predicts the whole population, so no sampling weights are needed for the final analysis.
- Reviews are never dropped for being short or repetitive. Only reviews that are empty after cleaning are excluded, and the count is reported.
- The labeling sheet hides the star rating and the train/test role, so labels depend on the text alone.
- Anomalies are printed as `FLAG` lines at run time instead of being resolved silently.

## 0. Setup

In [13]:
%pip install -q google-play-scraper

In [14]:
import json
import random
import re
import sys
import time
import unicodedata
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from google_play_scraper import Sort, app as fetch_app_info, reviews
from sklearn.metrics import cohen_kappa_score, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold

In [15]:
# ---- project
SEED = 42
APPS = {
    "BRImo": "id.co.bri.brimo",
    "myBCA": "com.bca.mybca.omni.android",
    "Livin_by_Mandiri": "id.bmri.livin",
    "wondr_by_BNI": "id.bni.wondr",
}
PERIOD_START = "2026-01-01"
PERIOD_END = "2026-08-31"          # inclusive, the whole day

# ---- scraping
LANG = "id"
COUNTRY = "id"
PAGE_SIZE = 200
SLEEP_SECONDS = 1.0
MAX_RETRIES = 5
BACKOFF_BASE = 2
STOP_MARGIN_DAYS = 2               # keep scraping this long after passing PERIOD_START
SOURCE_TZ = "UTC"                  # timezone of the naive timestamps returned by the scraper
TARGET_TZ = "Asia/Jakarta"

# ---- audit
GAP_RATIO = 0.25                   # a month below this share of the app's median month is flagged
SPIKE_RATIO = 3

# ---- text
MIN_CHARS = 3

# ---- sampling and labeling
BATCH_ID = "batch01"
TEST_PER_APP = 125
TRAIN_PER_APP = 250
MAX_COPIES_PER_TEXT = 2            # cap on identical texts in the training pool (test set is uncapped)
OVERWRITE_SAMPLE = False
N_FOLDS = 5
LABELS = {0: "negative", 1: "neutral", 2: "positive"}

# ---- storage
MOUNT_DRIVE = True

period_start = pd.Timestamp(PERIOD_START)
period_end_exclusive = pd.Timestamp(PERIOD_END) + pd.Timedelta(days=1)

In [16]:
IN_COLAB = "google.colab" in sys.modules
BASE_DIR = Path("mbanking_sentiment")
if IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/mbanking_sentiment")

RAW_DIR = BASE_DIR / "data" / "raw"
INTERIM_DIR = BASE_DIR / "data" / "interim"
LABEL_DIR = BASE_DIR / "data" / "labeling"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
for folder in (RAW_DIR, INTERIM_DIR, LABEL_DIR, PROCESSED_DIR):
    folder.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
print("Data directory:", BASE_DIR.resolve() if not IN_COLAB else BASE_DIR)

Mounted at /content/drive
Data directory: /content/drive/MyDrive/mbanking_sentiment


## 1. Scraping

Reviews are requested newest-first and the scan continues past `PERIOD_START` by `STOP_MARGIN_DAYS`, because Google Play does not allow jumping to a date. Out-of-period rows stay in the raw file and are filtered in Stage 2.

Behavior that matters for reproducibility:

- **Complete day boundaries.** `PERIOD_END` is inclusive for the whole day (`< PERIOD_END + 1 day`). Comparing against midnight would drop the last day.
- **Timezone.** The scraper returns naive timestamps in the timezone of the machine that ran it. They are converted from `SOURCE_TZ` to `TARGET_TZ` so month boundaries follow Indonesian time. The machine offset is logged and checked in Stage 2.
- **Coverage check.** Every run records whether the scan actually reached `PERIOD_START` and why it stopped. A scan that ends early is flagged instead of silently accepted.
- **Retries and resume.** Failed requests are retried with exponential backoff. An app whose raw file and log already exist is not scraped again.
- **Review date.** Google Play may show the date of the latest edit instead of the first post, so `review_date` is the displayed date.
- **No personal fields.** User names and avatars are not stored.

In [17]:
def to_target_tz(values):
    """Convert naive timestamps from SOURCE_TZ to naive TARGET_TZ wall-clock time."""
    stamps = pd.to_datetime(values)
    return stamps.dt.tz_localize(SOURCE_TZ).dt.tz_convert(TARGET_TZ).dt.tz_localize(None)


def fetch_page(app_id, token):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return reviews(
                app_id,
                lang=LANG,
                country=COUNTRY,
                sort=Sort.NEWEST,
                count=PAGE_SIZE,
                continuation_token=token,
            )
        except Exception as exc:
            wait = BACKOFF_BASE ** attempt
            print(f"  request failed ({type(exc).__name__}: {exc}); retry {attempt}/{MAX_RETRIES} in {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"request failed {MAX_RETRIES} times for {app_id}")


def scrape_app(app_id, period_start_text):
    """Scan newest-first until the period start (plus margin) is passed or reviews run out."""
    stop_before = pd.Timestamp(period_start_text) - pd.Timedelta(days=STOP_MARGIN_DAYS)
    records, token, n_requests = [], None, 0
    while True:
        batch, token = fetch_page(app_id, token)
        n_requests += 1
        records.extend(batch)
        if not batch:
            return records, {"n_requests": n_requests, "stop_reason": "empty_page"}
        oldest = to_target_tz(pd.Series([min(r["at"] for r in batch)])).iloc[0]
        if n_requests % 25 == 0:
            print(f"  requests: {n_requests} | reviews: {len(records)} | oldest in batch: {oldest.date()}")
        if oldest < stop_before:
            return records, {"n_requests": n_requests, "stop_reason": "period_start_passed"}
        if token is None:
            return records, {"n_requests": n_requests, "stop_reason": "token_exhausted"}
        time.sleep(SLEEP_SECONDS)


RAW_COLUMNS = {
    "reviewId": "review_id",
    "content": "review_text",
    "score": "rating",
    "at": "review_date",
    "reviewCreatedVersion": "review_created_version",
    "appVersion": "app_version",
    "thumbsUpCount": "thumbs_up",
}


def build_frame(records, app_name, app_id):
    frame = pd.DataFrame(records)
    for source_column in RAW_COLUMNS:
        if source_column not in frame.columns:
            frame[source_column] = None
    frame = frame[list(RAW_COLUMNS)].rename(columns=RAW_COLUMNS)
    frame["review_date"] = to_target_tz(frame["review_date"])
    frame["app_name"] = app_name
    frame["app_id"] = app_id
    return frame.sort_values("review_date").reset_index(drop=True)


def get_store_info(app_id):
    try:
        info = fetch_app_info(app_id, lang=LANG, country=COUNTRY)
        return {
            "store_title": info.get("title"),
            "store_version": info.get("version"),
            "store_ratings": info.get("ratings"),
            "store_reviews": info.get("reviews"),
        }
    except Exception as exc:
        return {
            "store_title": f"unavailable ({type(exc).__name__})",
            "store_version": None,
            "store_ratings": None,
            "store_reviews": None,
        }


def make_scrape_log(app_name, app_id, frame, run_info, scraped_at):
    oldest, newest = frame["review_date"].min(), frame["review_date"].max()
    log = {
        "app_name": app_name,
        "app_id": app_id,
        "scraped_at": scraped_at.isoformat(timespec="seconds"),
        "machine_utc_offset_hours": scraped_at.utcoffset().total_seconds() / 3600,
        "n_requests": run_info["n_requests"],
        "stop_reason": run_info["stop_reason"],
        "n_scraped": int(len(frame)),
        "oldest_review": str(oldest),
        "newest_review": str(newest),
        "reached_period_start": bool(pd.notna(oldest) and oldest <= period_start),
    }
    log.update(get_store_info(app_id))
    return log


def read_raw(path):
    return pd.read_csv(
        path,
        encoding="utf-8-sig",
        parse_dates=["review_date"],
        dtype={"review_id": str, "review_text": str, "review_created_version": str, "app_version": str},
    )

In [18]:
scrape_logs = []
raw_frames = {}

for app_name, app_id in APPS.items():
    raw_path = RAW_DIR / f"{app_name}_raw.csv"
    log_path = RAW_DIR / f"{app_name}_scrape_log.json"

    if raw_path.exists() and log_path.exists():
        print(f"{app_name}: existing raw file found, scraping skipped")
    else:
        print(f"{app_name} ({app_id}): scraping")
        scraped_at = datetime.now().astimezone()
        records, run_info = scrape_app(app_id, PERIOD_START)
        frame = build_frame(records, app_name, app_id)
        frame.to_csv(raw_path, index=False, encoding="utf-8-sig")
        log = make_scrape_log(app_name, app_id, frame, run_info, scraped_at)
        log_path.write_text(json.dumps(log, indent=2, ensure_ascii=False), encoding="utf-8")
        print(f"  saved {len(frame)} rows | stop reason: {run_info['stop_reason']}")

    raw_frames[app_name] = read_raw(raw_path)
    scrape_logs.append(json.loads(log_path.read_text(encoding="utf-8")))

BRImo: existing raw file found, scraping skipped
myBCA: existing raw file found, scraping skipped
Livin_by_Mandiri: existing raw file found, scraping skipped
wondr_by_BNI: existing raw file found, scraping skipped


## 2. Audit and Study Population

The audit prints coverage, missingness, duplicates, and text-quality diagnostics for the raw scrape and collects every anomaly as a `FLAG`. The study population is then built in four counted steps: scraped, inside the period, unique `review_id`, non-empty text.

In [19]:
def audit_population(df, logs):
    flags = []
    in_period = df["review_date"].between(period_start, period_end_exclusive, inclusive="left")

    print("1) Scrape coverage")
    log_table = pd.DataFrame(logs)[
        ["app_name", "store_title", "n_scraped", "oldest_review", "newest_review",
         "reached_period_start", "stop_reason", "n_requests"]
    ]
    print(log_table.to_string(index=False))
    for row in logs:
        if not row["reached_period_start"]:
            flags.append(f"{row['app_name']}: scan ended ({row['stop_reason']}) before reaching {PERIOD_START}")
        if SOURCE_TZ == "UTC" and row["machine_utc_offset_hours"] != 0:
            flags.append(f"{row['app_name']}: scraper machine offset is {row['machine_utc_offset_hours']}h, SOURCE_TZ is UTC")

    print("\n2) Reviews per app")
    per_app = pd.DataFrame({
        "scraped": df.groupby("app_name").size(),
        "in_period": df[in_period].groupby("app_name").size(),
    })
    print(per_app)

    print("\n3) Identifiers and empty text")
    n_dup_ids = int(df.duplicated("review_id").sum())
    n_blank = int((df["review_text"].isna() | df["review_text"].astype(str).str.strip().eq("")).sum())
    print(f"duplicated review_id rows: {n_dup_ids}")
    print(f"blank review_text rows: {n_blank}")
    if n_dup_ids:
        flags.append(f"{n_dup_ids} duplicated review_id rows")
    if n_blank:
        flags.append(f"{n_blank} reviews with blank text")

    inside = df[in_period].copy()
    inside["month"] = inside["review_date"].dt.to_period("M")
    expected_months = pd.period_range(period_start, PERIOD_END, freq="M")

    print("\n4) Reviews per app and month (inside the period)")
    month_table = inside.pivot_table(index="month", columns="app_name", values="review_id",
                                     aggfunc="count", fill_value=0).reindex(expected_months, fill_value=0)
    print(month_table)
    for app_name in month_table.columns:
        monthly = month_table[app_name]
        typical = monthly.median()
        for month, n_reviews in monthly.items():
            if n_reviews == 0 or n_reviews < GAP_RATIO * typical:
                flags.append(f"{app_name}: {month} has {n_reviews} reviews (monthly median {typical:.0f})")

    all_days = pd.date_range(period_start, PERIOD_END, freq="D")
    for app_name, app_df in inside.groupby("app_name"):
        days_present = pd.DatetimeIndex(app_df["review_date"].dt.normalize().unique())
        days_missing = all_days.difference(days_present)
        if len(days_missing):
            flags.append(f"{app_name}: {len(days_missing)} days without reviews (first: {days_missing[0].date()})")

    for month, n_reviews in monthly.items():
        if n_reviews > SPIKE_RATIO * typical:
            flags.append(f"{app_name}: {month} has {n_reviews} reviews (monthly median {typical:.0f})")

    print("\n5) Rating distribution (share within app)")
    print(pd.crosstab(inside["app_name"], inside["rating"], normalize="index").round(3))

    print("\n6) Missing app version (share within app)")
    print(inside.groupby("app_name")[["review_created_version", "app_version"]]
          .agg(lambda s: s.isna().mean()).round(3))

    print("\n7) Text quality")
    text_key = inside["review_text"].fillna("").str.strip().str.lower()
    short_share = text_key.str.len().lt(MIN_CHARS).groupby(inside["app_name"]).mean().round(4)
    duplicate_share = text_key.duplicated(keep=False).groupby(inside["app_name"]).mean().round(4)
    print(pd.DataFrame({"share_shorter_than_min_chars": short_share, "share_of_rows_with_repeated_text": duplicate_share}))
    print("\nMost frequent texts:")
    print(text_key.value_counts().head(10))

    print("\nFLAGS")
    if flags:
        for flag in flags:
            print(" - " + flag)
    else:
        print(" none")
    return flags

In [20]:
raw_all = pd.concat(raw_frames.values(), ignore_index=True)
audit_flags = audit_population(raw_all, scrape_logs)

in_period_mask = raw_all["review_date"].between(period_start, period_end_exclusive, inclusive="left")
in_period_df = raw_all[in_period_mask]
deduplicated = in_period_df.drop_duplicates("review_id")
population = deduplicated[deduplicated["review_text"].fillna("").str.strip() != ""].reset_index(drop=True)

waterfall = pd.DataFrame({
    "scraped": raw_all.groupby("app_name").size(),
    "in_period": in_period_df.groupby("app_name").size(),
    "unique_review_id": deduplicated.groupby("app_name").size(),
    "non_empty_text": population.groupby("app_name").size(),
})
waterfall.loc["total"] = waterfall.sum()
print("\nPopulation construction")
print(waterfall)

population.to_csv(INTERIM_DIR / "population_period.csv", index=False, encoding="utf-8-sig")

1) Scrape coverage
        app_name             store_title  n_scraped       oldest_review       newest_review  reached_period_start         stop_reason  n_requests
           BRImo                   BRImo      70400 2025-12-29 13:16:10 2026-10-30 19:24:54                  True period_start_passed         352
           myBCA myBCA: BCA Banking Apps      11000 2025-12-24 20:13:27 2026-10-30 18:38:49                  True period_start_passed          55
Livin_by_Mandiri       Livin' by Mandiri      41000 2025-12-28 16:06:13 2026-10-30 18:53:06                  True period_start_passed         205
    wondr_by_BNI            wondr by BNI      40000 2025-12-28 16:30:32 2026-10-30 19:46:31                  True period_start_passed         200

2) Reviews per app
                  scraped  in_period
app_name                            
BRImo               70400      59506
Livin_by_Mandiri    41000      35968
myBCA               11000       8832
wondr_by_BNI        40000      30604

3) Ident

## 3. Text Normalization

Three text columns are produced so that preprocessing can be compared empirically on identical folds:

| Column | Content | Use |
|---|---|---|
| `text_raw` | original text | configuration A (reference) |
| `text_norm` | light normalization | configuration B |
| `text_norm_slang` | `text_norm` plus abbreviation dictionary | configuration C |

Light normalization applies Unicode NFKC (for example fancy fonts to plain letters), removes zero-width characters, URLs and @mentions, collapses letters or symbols repeated three or more times to two (digits are untouched so amounts stay intact), and normalizes whitespace. Case and emoji are kept because the tokenizer handles case and emoji can carry sentiment. The abbreviation dictionary is deliberately small and limited to unambiguous forms; the hit counts below show which entries matter.

Quality flags are columns, not filters: `n_chars`, `n_words`, `is_short`, `has_emoji`, `is_emoji_only`, `text_freq` and `text_key`. `text_key` is the lowercase text without punctuation and is used to group identical reviews.

In [21]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")
ZERO_WIDTH_RE = re.compile("[\u200b\u200c\u200e\u200f\u202a-\u202e\u2060\ufeff]")
LETTER_REPEAT_RE = re.compile(r"([^\W\d_])\1{2,}")
SYMBOL_REPEAT_RE = re.compile(r"([^\w\s])\1{2,}")
SPACE_RE = re.compile(r"\s+")
EMOJI_RE = re.compile("[\U0001F000-\U0001FAFF\u2600-\u27BF\uFE0F]")

SLANG_MAP = {
    "gk": "tidak", "ga": "tidak", "gak": "tidak", "ngk": "tidak", "nggak": "tidak",
    "enggak": "tidak", "tdk": "tidak", "gabisa": "tidak bisa",
    "bgt": "banget", "bngt": "banget",
    "udh": "sudah", "sdh": "sudah", "udah": "sudah",
    "yg": "yang", "dgn": "dengan", "utk": "untuk",
    "krn": "karena", "karna": "karena", "krna": "karena",
    "jgn": "jangan", "trs": "terus", "trus": "terus",
    "jd": "jadi", "jdi": "jadi", "bs": "bisa", "bsa": "bisa",
    "knp": "kenapa", "knpa": "kenapa", "sy": "saya",
    "dr": "dari", "tp": "tapi", "tpi": "tapi",
    "blm": "belum", "blom": "belum", "lg": "lagi", "lgi": "lagi",
    "hrs": "harus", "org": "orang", "pake": "pakai", "apk": "aplikasi",
    "mantab": "mantap", "mantep": "mantap", "mantaf": "mantap", "mantul": "mantap",
    "bgus": "bagus", "bgs": "bagus",
    "eror": "error", "erorr": "error",
    "makasih": "terima kasih", "trims": "terima kasih",
    "gmn": "bagaimana", "gmna": "bagaimana", "gimana": "bagaimana",
    "kyk": "seperti", "kayak": "seperti",
    "klo": "kalau", "kalo": "kalau", "klw": "kalau",
    "skrg": "sekarang", "skrang": "sekarang", "msh": "masih", "dlm": "dalam",
    "pdhl": "padahal", "pdahal": "padahal", "sblm": "sebelum", "stlh": "setelah",
    "tlg": "tolong", "tf": "transfer", "trf": "transfer", "rek": "rekening",
}
SLANG_RE = re.compile(r"\b(" + "|".join(map(re.escape, sorted(SLANG_MAP, key=len, reverse=True))) + r")\b",
                      flags=re.IGNORECASE)


def normalize_light(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = ZERO_WIDTH_RE.sub("", text)
    text = URL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = LETTER_REPEAT_RE.sub(r"\1\1", text)
    text = SYMBOL_REPEAT_RE.sub(r"\1\1", text)
    return SPACE_RE.sub(" ", text).strip()


def normalize_slang(text):
    return SLANG_RE.sub(lambda match: SLANG_MAP[match.group(0).lower()], text)


def make_text_key(text):
    key = SPACE_RE.sub(" ", re.sub(r"[^\w\s]", "", text.lower())).strip()
    return key if key else text.lower()


def add_text_features(df):
    out = df.copy()
    out["text_raw"] = out["review_text"].fillna("").astype(str)
    out["text_norm"] = out["text_raw"].map(normalize_light)
    out["text_norm_slang"] = out["text_norm"].map(normalize_slang)
    out["text_key"] = out["text_norm"].map(make_text_key)
    out["n_chars"] = out["text_norm"].str.len()
    out["n_words"] = out["text_norm"].str.split().str.len()
    out["has_emoji"] = out["text_norm"].str.contains(EMOJI_RE)
    has_alnum = out["text_norm"].str.contains(r"[^\W_]", regex=True)
    out["is_emoji_only"] = out["has_emoji"] & ~has_alnum
    out["is_short"] = out["n_chars"] < MIN_CHARS
    out["text_freq"] = out.groupby("text_key")["text_key"].transform("size")
    out["review_month"] = out["review_date"].dt.to_period("M").astype(str)
    return out

In [22]:
population_norm = add_text_features(population)

n_empty_after_cleaning = int((population_norm["n_chars"] == 0).sum())
print(f"reviews empty after cleaning (excluded): {n_empty_after_cleaning}")
if n_empty_after_cleaning:
    print("FLAG: reviews that contained only URLs, mentions or invisible characters were excluded")
population_norm = population_norm[population_norm["n_chars"] > 0].reset_index(drop=True)

print(f"\npopulation size: {len(population_norm)}")
print(f"unique texts (text_key): {population_norm['text_key'].nunique()}")
print(f"rows changed by light normalization: {(population_norm['text_raw'] != population_norm['text_norm']).sum()}")
print(f"rows changed by abbreviation dictionary: {(population_norm['text_norm'] != population_norm['text_norm_slang']).sum()}")

print("\nShare of rows by app")
quality = population_norm.groupby("app_name").agg(
    short=("is_short", "mean"),
    emoji_only=("is_emoji_only", "mean"),
    has_emoji=("has_emoji", "mean"),
    repeated_text=("text_freq", lambda s: (s > 1).mean()),
).round(4)
print(quality)

hits = population_norm["text_norm"].str.lower().str.findall(SLANG_RE).explode().dropna().value_counts()
print("\nMost frequent dictionary hits")
print(hits.head(15))

print("\nExamples of changed texts")
changed = population_norm[population_norm["text_raw"] != population_norm["text_norm_slang"]]
print(changed[["text_raw", "text_norm", "text_norm_slang"]].sample(8, random_state=SEED).to_string(index=False))

BLOCKING_PATTERN = "aktivis|aktifis|boikot|sepihak|pemblokiran|main blokir|tukang blokir|penegak hukum|botok"
population_norm["blocking_topic"] = population_norm["text_norm"].str.lower().str.contains(BLOCKING_PATTERN)
print(pd.crosstab(population_norm["review_month"], [population_norm["app_name"], population_norm["blocking_topic"]]))

population_norm.to_csv(PROCESSED_DIR / "population_normalized.csv", index=False, encoding="utf-8-sig")

reviews empty after cleaning (excluded): 0

population size: 134910
unique texts (text_key): 82290
rows changed by light normalization: 12716
rows changed by abbreviation dictionary: 33118

Share of rows by app
                   short  emoji_only  has_emoji  repeated_text
app_name                                                      
BRImo             0.0342      0.0087     0.0605         0.4512
Livin_by_Mandiri  0.0206      0.0066     0.0594         0.3099
myBCA             0.0288      0.0046     0.0580         0.2360
wondr_by_BNI      0.0460      0.0112     0.0572         0.5314

Most frequent dictionary hits
text_norm
gak       7729
ga        6772
yg        4983
udah      4516
apk       3933
gk        2984
pake      2962
eror      1927
kalo      1802
gimana    1710
udh       1334
tf        1259
bgt       1240
gabisa    1210
tdk       1121
Name: count, dtype: int64

Examples of changed texts
                                                                                            

## 4. Sampling for Manual Labeling

Two disjoint samples are drawn from the normalized population:

- **Test set.** Simple random sample, `TEST_PER_APP` reviews per app, with no cap on repeated texts. It mirrors what the model will meet in the full population and supports per-app reporting.
- **Training pool.** `TRAIN_PER_APP` reviews per app with equal quotas across months, and at most `MAX_COPIES_PER_TEXT` identical texts in the whole pool. Frequent one-word reviews therefore do not crowd out informative ones.

Identical texts may occur in both samples. `text_shared_across_roles` marks them so test results can be reported with and without them.

The labeling sheet contains only `sample_id`, `app_name` and `text_raw`; rating, date and role stay in the key file. Rows are shuffled before IDs are assigned. If the key file for the batch already exists it is loaded instead of redrawn, which protects labels already made against a changed population.

In [23]:
def draw_test_set(pop, per_app, seed):
    parts = [group.sample(n=min(per_app, len(group)), random_state=seed) for _, group in pop.groupby("app_name")]
    return pd.concat(parts)


def draw_train_pool(pop, exclude_ids, per_app, max_copies, seed):
    candidates = pop[~pop["review_id"].isin(exclude_ids)].copy()
    candidates["month"] = candidates["review_date"].dt.to_period("M")
    taken, shortfalls, copies = [], [], {}
    for app_name, app_df in candidates.groupby("app_name"):
        months = sorted(app_df["month"].unique())
        base, extra = divmod(per_app, len(months))
        for i, month in enumerate(months):
            quota = base + (1 if i < extra else 0)
            shuffled = app_df[app_df["month"] == month].sample(frac=1, random_state=seed + i)
            chosen = []
            for index, key in zip(shuffled.index, shuffled["text_key"]):
                if copies.get(key, 0) >= max_copies:
                    continue
                copies[key] = copies.get(key, 0) + 1
                chosen.append(index)
                if len(chosen) == quota:
                    break
            taken.extend(chosen)
            if len(chosen) < quota:
                shortfalls.append({"app_name": app_name, "month": str(month), "quota": quota, "drawn": len(chosen)})
    return pop.loc[taken], pd.DataFrame(shortfalls)


KEY_PATH = LABEL_DIR / f"sample_key_{BATCH_ID}.csv"
SHEET_PATH = LABEL_DIR / f"labeling_sheet_{BATCH_ID}.csv"
KEY_COLUMNS = [
    "sample_id", "role", "review_id", "app_name", "review_date", "rating", "thumbs_up",
    "review_created_version", "app_version", "text_raw", "text_norm", "text_norm_slang",
    "text_key", "text_freq", "n_chars", "n_words", "has_emoji", "is_emoji_only", "is_short",
    "text_shared_across_roles",
]

if KEY_PATH.exists() and not OVERWRITE_SAMPLE:
    print(f"{KEY_PATH.name} exists: loaded, not redrawn")
    sample_key = pd.read_csv(KEY_PATH, encoding="utf-8-sig", parse_dates=["review_date"],
                             dtype={"review_created_version": str, "app_version": str})
else:
    previous_ids = set()
    for other_key_path in LABEL_DIR.glob("sample_key_*.csv"):
        if other_key_path != KEY_PATH:
            previous_ids |= set(pd.read_csv(other_key_path, encoding="utf-8-sig", usecols=["review_id"])["review_id"])
    print(f"review_id already used in earlier batches: {len(previous_ids)}")

    available = population_norm[~population_norm["review_id"].isin(previous_ids)]
    test_set = draw_test_set(available, TEST_PER_APP, SEED)
    train_pool, train_shortfalls = draw_train_pool(
        available, set(test_set["review_id"]), TRAIN_PER_APP, MAX_COPIES_PER_TEXT, SEED)

    sample_key = pd.concat([test_set.assign(role="test"), train_pool.assign(role="train_pool")])
    sample_key = sample_key.sample(frac=1, random_state=SEED).reset_index(drop=True)
    sample_key.insert(0, "sample_id", [f"S{i + 1:05d}" for i in range(len(sample_key))])
    roles_per_text = sample_key.groupby("text_key")["role"].nunique()
    sample_key["text_shared_across_roles"] = sample_key["text_key"].map(roles_per_text) > 1
    sample_key = sample_key[KEY_COLUMNS]

    sample_key.to_csv(KEY_PATH, index=False, encoding="utf-8-sig")
    sheet = sample_key[["sample_id", "app_name", "text_raw"]].assign(label_id="", uncertain="")
    sheet.to_csv(SHEET_PATH, index=False, encoding="utf-8-sig")
    print(f"saved {KEY_PATH.name} and {SHEET_PATH.name}")

    if len(train_shortfalls):
        print("FLAG: training-pool quotas not met (duplicate cap or small month):")
        print(train_shortfalls.to_string(index=False))

print("\nSample size by role and app")
print(pd.crosstab(sample_key["app_name"], sample_key["role"], margins=True))
print("\nTraining pool by app and month")
print(pd.crosstab(sample_key.loc[sample_key["role"] == "train_pool", "app_name"],
                  sample_key.loc[sample_key["role"] == "train_pool", "review_date"].dt.to_period("M")))

print("\nSample versus population (share of rows)")
comparison = pd.DataFrame({
    "population": [population_norm["is_short"].mean(), (population_norm["text_freq"] > 1).mean()],
    "test": [sample_key.loc[sample_key["role"] == "test", "is_short"].mean(),
             (sample_key.loc[sample_key["role"] == "test", "text_freq"] > 1).mean()],
    "train_pool": [sample_key.loc[sample_key["role"] == "train_pool", "is_short"].mean(),
                   (sample_key.loc[sample_key["role"] == "train_pool", "text_freq"] > 1).mean()],
}, index=["is_short", "text_repeated_in_population"]).round(4)
print(comparison)
print(f"\nrows whose text also appears in the other role: {int(sample_key['text_shared_across_roles'].sum())}")

sample_key_batch01.csv exists: loaded, not redrawn

Sample size by role and app
role              test  train_pool   All
app_name                                
BRImo              125         250   375
Livin_by_Mandiri   125         250   375
myBCA              125         250   375
wondr_by_BNI       125         250   375
All                500        1000  1500

Training pool by app and month
review_date       2026-01  2026-02  2026-03  2026-04  2026-05  2026-06  \
app_name                                                                 
BRImo                  32       32       31       31       31       31   
Livin_by_Mandiri       32       32       31       31       31       31   
myBCA                  32       32       31       31       31       31   
wondr_by_BNI           32       32       31       31       31       31   

review_date       2026-07  2026-08  
app_name                            
BRImo                  31       31  
Livin_by_Mandiri       31       31  
myBCA   

## 5. Labeling Guidelines

Classes: **0 = negative**, **1 = neutral**, **2 = positive**. Labels depend on the review text only. The star rating is never shown in the labeling sheet and is used afterwards as an audit signal.

Rule status: *agreed* means decided jointly with the project owner; *convention* means applied by the labeler to keep decisions consistent and open to revision.

| # | Rule | Label | Examples | Status |
|---|---|---|---|---|
| 1 | Explicit satisfaction, praise, recommendation or gratitude toward the app or bank; positive emoji only | positive | "mantap", "sangat membantu", "terima kasih BRImo", 👍 | convention for gratitude |
| 2 | Complaint, failure report, frustration or insult: app does not open, login fails, error or lag, failed or lost transaction, unexpected fees, blocked account. Questions that state a failure count here | negative | "kenapa aplikasi tidak bisa dibuka?", "saldo terpotong tapi transaksi gagal" | agreed |
| 3 | "ok" or "oke" standing alone, with no other evaluative word | neutral | "ok", "oke" | agreed |
| 3b | "ok" together with an evaluative word | positive | "oke banget", "ok bagus" | convention |
| 4 | Procedural or situational question without a stated app failure; information question | neutral | "bagaimana login kalau lupa sandi?", "kapan pemenang diumumkan?" | agreed |
| 5 | Praise followed by a concrete complaint | negative | "bagus tapi sering gagal", "keren, cuma updatenya terlalu sering" | agreed |
| 5b | Praise followed by a vague wish for improvement without a concrete defect | positive | "mantap, keamanan perlu ditingkatkan" | convention |
| 6 | Suggestion or feature request without complaint tone | neutral | "tolong tambahkan fitur hapus riwayat VA" | convention |
| 6b | Request that describes a concrete flaw | negative | "PIN tertangkap Samsung Pass, tingkatkan keamanan" | agreed |
| 7 | Well wishes addressed to the app or bank | positive | "sukses selalu" | convention |
| 7b | Hope about the reviewer's own outcome, with no evaluation | neutral | "semoga aman", "semoga dimudahkan" | convention |
| 8 | Statement without evaluation, test text, single numerals, off-topic or unintelligible text | neutral, `uncertain = 1` for unintelligible | "sudah bertahun-tahun menggunakan", "coba2", "1" | convention |
| 9 | Sarcasm: label the intended meaning | usually negative, `uncertain = 1` | "baru tanggal 1 sudah dipotong admin... mantap" | convention |
| 10 | Comparison with another app: label the sentiment toward the reviewed app | by sentiment | "tak selancar BCA mobile" → negative | convention |
| 11 | Review about a related product (ATM, card, branch, prize draw) | by sentiment as written | | convention |
| 12 | Identical texts receive identical labels | | checked in Stage 6 | rule |
| 13 | Borderline decision or two rules in conflict | `uncertain = 1` | | rule |
Varian ejaan “ok” yang berdiri sendiri → netral; kalau ada kata evaluatif (“ok manta9”, “ok bajk”) → positif.
Pernyataan mencoba atau pengguna baru (“baru coba”, “coba dulu”) → netral. Menyebut “bintang 5” → positif uncertain; “bintang 4 dulu” → netral uncertain.
Permintaan yang menyiratkan cacat kinerja (“tingkatkan penghematan RAM”, “percepat lagi untuk login”, “harus lebih baik”) → negatif uncertain. Permintaan fitur murni tetap netral.

**Sheet format.** The labeling sheet has `sample_id`, `app_name`, `text_raw`, `label_id`, `uncertain`. The completed file is saved as `labeled_<batch>.csv` in the labeling folder and needs at least `sample_id`, `label_id` (0, 1 or 2) and `uncertain` (0 or 1).

**Agreement check.** A second annotator labels a random subset plus every row marked uncertain in a column named `label_id_human`. Agreement is reported separately for the random subset, which estimates label quality, and for uncertain rows, which are expected to show lower agreement.

In [24]:
pattern = "aktivis|aktifis|boikot|sepihak|pemblokiran|blokir rekening"
livin = sample_key[sample_key["app_name"] == "Livin_by_Mandiri"]
print(pd.crosstab(livin["review_date"].dt.to_period("M"), livin["text_norm"].str.lower().str.contains(pattern)))

text_norm    False  True 
review_date              
2026-01         37      0
2026-02         44      0
2026-03         45      0
2026-04         40      0
2026-05         38      0
2026-06         36      0
2026-07         43      0
2026-08         66     26


## 6. Label Validation

Runs once the completed labeled file exists. The check reads the sample key from disk, so it also works after a runtime restart. It prints a `FLAG` for every inconsistency and does not change any label.

In [25]:
LABELED_PATH = LABEL_DIR / f"labeled_{BATCH_ID}.csv"
OK_KEYS = {"ok", "oke", "okee", "okey", "ok lah", "oke lah"}


def validate_labels(labeled, key):
    flags = []
    missing_columns = {"sample_id", "label_id", "uncertain"} - set(labeled.columns)
    if missing_columns:
        raise ValueError(f"labeled file lacks columns: {sorted(missing_columns)}")

    if labeled["sample_id"].duplicated().any():
        flags.append(f"{int(labeled['sample_id'].duplicated().sum())} duplicated sample_id rows in the labeled file")
    unknown_ids = set(labeled["sample_id"]) - set(key["sample_id"])
    if unknown_ids:
        flags.append(f"{len(unknown_ids)} sample_id values are not in the sample key")
    unlabeled_ids = set(key["sample_id"]) - set(labeled["sample_id"])
    if unlabeled_ids:
        flags.append(f"{len(unlabeled_ids)} sampled reviews have no row in the labeled file")

    merged = key.merge(labeled.drop_duplicates("sample_id")[["sample_id", "label_id", "uncertain"]],
                       on="sample_id", how="inner", validate="one_to_one")
    valid_label = merged["label_id"].isin(list(LABELS))
    if (~valid_label).any():
        flags.append(f"{int((~valid_label).sum())} rows with a missing or invalid label_id (excluded below)")
    merged = merged[valid_label].copy()
    merged["label_id"] = merged["label_id"].astype(int)
    merged["uncertain"] = merged["uncertain"].fillna(0).astype(int)
    merged["label"] = merged["label_id"].map(LABELS)

    labels_per_text = merged.groupby("text_key")["label_id"].nunique()
    conflicts = labels_per_text[labels_per_text > 1].index
    if len(conflicts):
        flags.append(f"{len(conflicts)} identical texts received different labels")
        print("Texts with conflicting labels (first 20)")
        shown = merged[merged["text_key"].isin(conflicts[:20])].sort_values("text_key")
        print(shown.assign(text_raw=shown["text_raw"].str.slice(0, 70))[["sample_id", "text_raw", "label"]]
              .to_string(index=False))

    discordant = merged[((merged["rating"] >= 4) & (merged["label_id"] == 0)) |
                        ((merged["rating"] <= 2) & (merged["label_id"] == 2))]
    print(f"\nrating-label discordance: {len(discordant)} rows (not errors by default)")
    shown = discordant.head(15)
    print(shown.assign(text_raw=shown["text_raw"].str.slice(0, 70))[["sample_id", "rating", "label", "text_raw"]]
          .to_string(index=False))

    print("\nLabel distribution by app")
    by_app = pd.crosstab(merged["app_name"], merged["label"], margins=True)
    print(by_app)
    print("\nLabel distribution by role")
    print(pd.crosstab(merged["role"], merged["label"], margins=True))

    neutral = merged[merged["label_id"] == 1]
    print(f"\nneutral rows: {len(neutral)} | unique texts: {neutral['text_key'].nunique()} | "
          f"'ok'-type texts: {int(neutral['text_key'].isin(OK_KEYS).sum())}")
    if len(neutral) < 0.10 * len(merged):
        flags.append(f"neutral class is {len(neutral) / len(merged):.1%} of labeled rows")
    sparse = [(app, label) for app in by_app.index[:-1] for label in by_app.columns[:-1] if by_app.loc[app, label] < 10]
    for app, label in sparse:
        flags.append(f"{app}: only {by_app.loc[app, label]} {label} rows")

    print(f"\nuncertain rows: {int(merged['uncertain'].sum())}")
    print("\nFLAGS")
    if flags:
        for flag in flags:
            print(" - " + flag)
    else:
        print(" none")
    return merged, flags


def agreement_report(df, column_a, column_b):
    both = df.dropna(subset=[column_a, column_b])
    a, b = both[column_a].astype(int), both[column_b].astype(int)
    print(f"n = {len(both)} | agreement = {(a == b).mean():.3f} | "
          f"kappa = {cohen_kappa_score(a, b):.3f} | weighted kappa = {cohen_kappa_score(a, b, weights='linear'):.3f}")
    print(pd.DataFrame(confusion_matrix(a, b, labels=list(LABELS)),
                       index=[f"A_{name}" for name in LABELS.values()],
                       columns=[f"B_{name}" for name in LABELS.values()]))
    return both


labeled_merged = None
if LABELED_PATH.exists():
    BATCH_IDS = ["batch01", "batch02"]
    sample_key_disk = pd.concat(
        [pd.read_csv(LABEL_DIR / f"sample_key_{batch}.csv", encoding="utf-8-sig", parse_dates=["review_date"],
                     dtype={"review_created_version": str, "app_version": str}) for batch in BATCH_IDS],
        ignore_index=True)
    labeled_sheet = pd.concat(
        [pd.read_csv(LABEL_DIR / f"labeled_{batch}.csv", encoding="utf-8-sig") for batch in BATCH_IDS],
        ignore_index=True)
    labeled_merged, label_flags = validate_labels(labeled_sheet, sample_key_disk)

    if "label_id_human" in labeled_sheet.columns:
        with_human = labeled_merged.merge(labeled_sheet[["sample_id", "label_id_human"]], on="sample_id")
        print("\nAgreement, uncertain rows excluded (estimate of label quality)")
        agreement_report(with_human[with_human["uncertain"] == 0], "label_id", "label_id_human")
        print("\nAgreement, uncertain rows only")
        agreement_report(with_human[with_human["uncertain"] == 1], "label_id", "label_id_human")
else:
    print(f"{LABELED_PATH.name} not found: Stages 6 and 7 are skipped until it exists")


rating-label discordance: 159 rows (not errors by default)
sample_id  rating    label                                                               text_raw
   S00017       4 negative kok makin hari makin lemot ya kalau mau buka ada 2 menit sendiri kalau
   S00045       5 negative                                      kenapa habis d update malah lemot
   S00056       5 negative habis di update gak bisa login lewat fingerprint. pengaturan login fin
   S00068       5 negative sudah ada update dari bca dan tidak lemot lagi terima kasih , tlg info
   S00075       5 negative             tarik tunai lewat atm, saldo maksimal ditambah jangan 1jt.
   S00091       5 negative                                          kenapa jadi lemot ya buka nya
   S00097       1 positive                                                                  bagus
   S00126       5 negative                                   tidak bisa di pakai sekarang akunnya
   S00142       5 negative kok saya tf ke rekening BNI dar

## 7. Training Data Assembly

Training-pool rows receive a cross-validation fold from `StratifiedGroupKFold`, stratified on the label and grouped on `text_key`, so identical texts never fall in different folds. Test rows keep `fold = -1`. The final files keep every text version (`text_raw`, `text_norm`, `text_norm_slang`) so preprocessing configurations can be compared without rebuilding the data.

In [26]:
def assign_folds(train_df, n_splits, seed):
    out = train_df.reset_index(drop=True).copy()
    out["fold"] = -1
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for fold, (_, valid_index) in enumerate(splitter.split(out, out["label_id"], groups=out["text_key"])):
        out.loc[valid_index, "fold"] = fold
    return out


FINAL_COLUMNS = [
    "sample_id", "review_id", "app_name", "review_date", "review_created_version", "app_version",
    "rating", "thumbs_up", "text_raw", "text_norm", "text_norm_slang", "text_key",
    "label_id", "label", "uncertain", "role", "text_shared_across_roles", "fold",
]

FINAL_COLUMNS.append("selection_source")

if labeled_merged is None:
    print("No validated labels available: Stage 7 skipped")
else:
    labeled_merged["selection_source"] = labeled_merged["selection_source"].fillna("batch01")
    train_part = assign_folds(labeled_merged[labeled_merged["role"] == "train_pool"], N_FOLDS, SEED)
    test_part = labeled_merged[labeled_merged["role"] == "test"].reset_index(drop=True).assign(fold=-1)

    leaking_texts = int((train_part.groupby("text_key")["fold"].nunique() > 1).sum())
    print(f"texts spread over more than one fold: {leaking_texts}")
    if leaking_texts:
        print("FLAG: grouped folds are leaking identical texts")

    print("\nLabel counts per fold (training pool)")
    print(pd.crosstab(train_part["fold"], train_part["label"]))
    print("\nApp counts per fold (training pool)")
    print(pd.crosstab(train_part["fold"], train_part["app_name"]))

    train_part[FINAL_COLUMNS].to_csv(PROCESSED_DIR / "train_pool.csv", index=False, encoding="utf-8-sig")
    test_part[FINAL_COLUMNS].to_csv(PROCESSED_DIR / "test.csv", index=False, encoding="utf-8-sig")
    print(f"\nsaved train_pool.csv ({len(train_part)} rows) and test.csv ({len(test_part)} rows)")

texts spread over more than one fold: 0

Label counts per fold (training pool)
label  negative  neutral  positive
fold                              
0           126       53       107
1           130       60        93
2           122       48       105
3           125       45       106
4           128       48       104

App counts per fold (training pool)
app_name  BRImo  Livin_by_Mandiri  myBCA  wondr_by_BNI
fold                                                  
0            65                67     81            73
1            63                71     72            77
2            70                70     65            70
3            76                72     64            64
4            76                70     68            66

saved train_pool.csv (1400 rows) and test.csv (500 rows)


## Data Dictionary

| Column | Description |
|---|---|
| `review_id` | Google Play review identifier |
| `app_name`, `app_id` | app label and Play Store package name |
| `review_date` | displayed review date, naive wall-clock time in `TARGET_TZ` |
| `review_created_version`, `app_version` | app version fields returned by the scraper; both can be missing |
| `rating` | star rating, 1 to 5 |
| `thumbs_up` | helpful votes |
| `text_raw`, `text_norm`, `text_norm_slang` | original, lightly normalized, and abbreviation-normalized text |
| `text_key` | lowercase text without punctuation, used to group identical reviews |
| `text_freq` | number of population rows sharing the same `text_key` |
| `n_chars`, `n_words`, `is_short`, `has_emoji`, `is_emoji_only` | text quality flags |
| `review_month` | month of `review_date` |
| `sample_id` | batch-specific identifier shown to the labeler |
| `role` | `test` or `train_pool` |
| `text_shared_across_roles` | the same `text_key` occurs in both the test set and the training pool |
| `label_id`, `label` | 0 negative, 1 neutral, 2 positive |
| `uncertain` | 1 when the labeling decision was borderline |
| `fold` | cross-validation fold of a training-pool row, `-1` for test rows |

cek selisih

In [19]:
disagree = with_human[with_human["label_id"] != with_human["label_id_human"]].dropna(subset=["label_id_human"])
print(disagree[["sample_id", "uncertain", "label", "label_id_human", "text_raw"]].to_string(index=False))

sample_id  uncertain    label  label_id_human                                                                                                                                                                                                                                                                                                                                                                                                                                                text_raw
   S00011          1 negative             1.0                                                                                                                                                                                                                                                                                                                                                                                                                     kenapa tidak otomatis login lagi ia
   S00068          1 negativ